In [1]:
import pandas as pd
import os
import numpy as np

file_path = os.path.join("tables", "exerciseTableForBN_python.parquet")
exerciseTable = pd.read_parquet(file_path)
exerciseTableDiscrete = exerciseTable.copy()

colsToRemove = ["PtID", "DeviceDtTm", "UTCDtTm", "ExerciseName", "DistanceValue", "DistanceUnits", "DurationUnits", "EnergyValue",
                 "EnergyUnits", "TmZnOffset", "CleanActivityName", "AOB", "CWL", "TimeSinceLastBolus", "LastBolus", 
                 "TimeSinceLastBasal", "LastBasal"]
exerciseTableDiscrete.drop(columns=colsToRemove, inplace=True)
# ----------------------- fixed layer -----------------------

# AGE: discretize into 3 quantiles
exerciseTableDiscrete["age"] = pd.qcut(exerciseTableDiscrete["age"], q=3, labels=["Younger", "Middle", "Older"])

# gender: 1 = male, 2 = female
exerciseTableDiscrete["gender"] = exerciseTable["gender"].map({1.0: "male", 2.0: "female"})

# HEIGHT: discretize into 3 quantiles
exerciseTableDiscrete["height"] = pd.qcut(exerciseTableDiscrete["height"], q=3, labels=["Shorter", "Average", "Taller"])

# WEIGHT: discretize into 3 quantiles
exerciseTableDiscrete["weight"] = pd.qcut(exerciseTableDiscrete["weight"], q=3, labels=["Lighter", "Average", "Heavier"])

# BMI: standard WHO categories
bmi_edges = [0, 18.5, 24.9, 29.9, np.inf]
bmi_labels = ["Underweight", "Normal", "Overweight", "Obese"]
exerciseTableDiscrete["BMI"] = pd.cut(exerciseTableDiscrete["BMI"], bins=bmi_edges, labels=bmi_labels)

# HbA1c: ADA diabetes care categories
hba1c_edges = [0, 5.7, 6.4, 7, 8, np.inf]
hba1c_labels = ["Normal", "Prediabetes", "Diabetes - Control", "Diabetes - Poor Control", "Diabetes - Very Poor Control"]
exerciseTableDiscrete["HbA1c"] = pd.cut(exerciseTableDiscrete["HbA1c"], bins=hba1c_edges, labels=hba1c_labels)

# Insulin Sensitivity: discretize into 3 quantiles
exerciseTableDiscrete["InsSensitivity"] = pd.qcut(exerciseTableDiscrete["InsSensitivity"], q=3, labels=["Low", "Medium", "High"])

# Insulin Carb Ratio: discretize into 3 quantiles
exerciseTableDiscrete["InsCarbRatio"] = pd.qcut(exerciseTableDiscrete["InsCarbRatio"], q=3, labels=["Low", "Medium", "High"])


# ------------------- pre exercise layer --------------------

# start glucose level: standard T1D categories
startGlucose_edges = [0, 70, 180, 250, np.inf]
startGlucose_labels = ["Hypo Risk", "Target", "Elevated", "Very High"]
exerciseTableDiscrete["startExerciseGlucoseLevel"] = pd.cut(exerciseTableDiscrete["startExerciseGlucoseLevel"], bins=startGlucose_edges, labels=startGlucose_labels)

# glucose rate of change: standard CGM trend categories
startRoc_edges = [-np.inf, -2, -1, 1, 2, np.inf]
startRoc_labels = ["Rapidly Falling", "Falling", "Stable", "Rising", "Rapidly Rising"]
exerciseTableDiscrete["preExerciseRoc"] = pd.cut(exerciseTableDiscrete["preExerciseRoc"], bins=startRoc_edges, labels=startRoc_labels)

# glucose CV: international consensus categories
startCV_edges = [0, 36, np.inf]
startCV_labels = ["Stable", "Unstable"]
exerciseTableDiscrete["preExerciseGlucoseCV"] = pd.cut(exerciseTableDiscrete["preExerciseGlucoseCV"], bins=startCV_edges, labels=startCV_labels)

# Insulin on Board: discretize into 4 quantiles
iob_edges = [-np.inf, -0.2, 0.2, 2, np.inf]
iob_labels = ["Negative", "Baseline", "Moderate", "High"]
exerciseTableDiscrete["IOB"] = pd.cut(exerciseTableDiscrete["IOB"], bins=iob_edges, labels=iob_labels)

# Carbohydrates on Board: discretize into 4 quantiles
exerciseTableDiscrete["COB"] = pd.qcut(exerciseTableDiscrete["COB"], q=4, labels=["Low", "Moderate", "High", "Very High"])
# cob_edges = [0, 20, 50, np.inf]
# cob_labels = ["Low", "Moderate", "High"]

# ACWR: discretize into 4 quantiles
exerciseTableDiscrete["ACWR"] = pd.qcut(exerciseTableDiscrete["ACWR"], q=4, labels=["Sedentary", "Lightly Active", "Active", "Highly Active"])


# ------------------- exercise layer --------------------

# MET: standard metabolic equivalent categories
met_edges = [0, 3, 6, np.inf]
met_labels = ["Light", "Moderate", "Vigorous"]
exerciseTableDiscrete["MET"] = pd.cut(exerciseTableDiscrete["MET"], bins=met_edges, labels=met_labels)

# duration: typical exercise duration categories
duration_edges = [0, 30, 60, np.inf]
duration_labels = ["Short", "Medium", "Long"]
exerciseTableDiscrete["DurationValue"] = pd.cut(exerciseTableDiscrete["DurationValue"], bins=duration_edges, labels=duration_labels)

# MET * min: discretize into 3 quantiles
exerciseTableDiscrete["MET_min"] = pd.qcut(exerciseTableDiscrete["MET_min"], q=3, labels=["Low", "Medium", "High"])

# energy per min (kcal/min): discretize into 3 quantiles
exerciseTableDiscrete["EnergyPerMinute"] = pd.qcut(exerciseTableDiscrete["EnergyPerMinute"], q=3, labels=["Low", "Medium", "High"])

# ------------------- outcome layer --------------------

# glucose excursion during exercise: discretize into 4 quantiles
exerciseGlucoseExcursion_edges = [-np.inf, -40, -10, 10, np.inf]
exerciseGlucoseExcursion_labels = ["Severe Drop", "Moderate Drop", "Stable", "Rise"]
exerciseTableDiscrete["exerciseGlucoseExcursion"] = pd.cut(exerciseTableDiscrete["exerciseGlucoseExcursion"], bins=exerciseGlucoseExcursion_edges, labels=exerciseGlucoseExcursion_labels)

# glucose rate of change during exercise: standard CGM trend categories (same as pre-exercise)
exerciseTableDiscrete["exerciseGlucoseRoc"] = pd.cut(exerciseTableDiscrete["exerciseGlucoseRoc"], bins=startRoc_edges, labels=startRoc_labels)

# minimum glucose value after exercise: standard T1D categories 
minGlucosePostExercise_edges = [-np.inf, 54, 70, 180, np.inf]
minGlucosePostExercise_labels = ["Severe Hypo", "Hypo Risk", "Target", "Elevated"]
exerciseTableDiscrete["minGlucosePostExercise"] = pd.cut(exerciseTableDiscrete["minGlucosePostExercise"], bins=minGlucosePostExercise_edges, labels=minGlucosePostExercise_labels)

# glucose CV after exercise: international consensus categories (same as pre-exercise)
exerciseTableDiscrete["postExerciseGlucoseCV"] = pd.cut(exerciseTableDiscrete["postExerciseGlucoseCV"], bins=startCV_edges, labels=startCV_labels)

# time in target range (70-180 mg/dL) after exercise: ADA categories
postExerciseTIR_edges = [0, 50, 70, np.inf]
postExerciseTIR_labels = ["Poor", "Borderline", "Target"]
exerciseTableDiscrete["postExerciseTIR"] = pd.cut(exerciseTableDiscrete["postExerciseTIR"], bins=postExerciseTIR_edges, labels=postExerciseTIR_labels)
exerciseTableDiscrete.to_parquet("exerciseTableDiscrete.parquet")